# Phishing Website Detection — 03: Features and Splits

This notebook decides what the paper actually claims. Modelling is the easy part;
the split design is what a reviewer will attack.

**What happens here**
1. Leakage audit — single-feature AUC sweep across every dataset
2. A canonical URL-lexical feature space, computed identically for every source
3. Three split strategies, with the leakage gap between them measured
4. A cross-dataset train/test harness
5. Benign-source bias probe against Tranco rank
6. A realistic base-rate evaluation set

### A correction on chronological splits

Earlier I recommended a chronological split. **None of these six tabular
benchmarks ship a timestamp column** — they are feature matrices extracted once,
in 2012, 2018, 2020 or 2023, with no per-row collection date. A "chronological"
split over them would be fabricated ordering.

So the temporal argument has to be carried differently:

- **Domain-grouped splitting** handles the near-duplicate problem within a dataset
  (phishing kits reuse hosts, so random splits put sibling URLs on both sides).
- **Cross-dataset evaluation** carries the generalisation burden. Training on a
  2023 source and testing on a 2020 source is a genuine distribution shift, and
  it is defensible in a way an invented date ordering is not.

If you want a true temporal experiment later, it requires collecting dated URLs
from OpenPhish over some weeks. That is a different paper — say so in Limitations
rather than faking it here.

## Cell 1 — Environment (self-contained)

In [1]:
!pip install -q ucimlrepo tldextract scikit-learn pandas numpy matplotlib

from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive')

PROJECT   = "phishing-detection"
REPO_DIR  = Path("/content") / PROJECT
DRIVE_DIR = Path("/content/drive/MyDrive/research") / PROJECT

for d in ["data/raw", "data/external", "data/interim", "data/processed",
          "checkpoints", "models", "logs", "notebooks"]:
    (DRIVE_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/data", "src/features", "src/models", "src/evaluation",
          "src/utils", "config", "reports", "reports/tables", "reports/figures"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/data", "src/features", "src/models", "src/evaluation", "src/utils"]:
    (REPO_DIR / d / "__init__.py").touch()

for name in ["data", "checkpoints", "models", "logs"]:
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f"{link} exists and is not a symlink")
    link.symlink_to(DRIVE_DIR / name)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

RAW = REPO_DIR / "data" / "raw"
print("raw ->", RAW.resolve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 4.4 MB/s eta 0:00:00
Mounted at /content/drive
raw -> /content/drive/MyDrive/research/phishing-detection/data/raw


## Cell 1b — Restore the checkpoint module

In [2]:
CKPT_SRC = 'import json, time, hashlib, subprocess\nfrom pathlib import Path\nimport joblib\n\n\ndef _git_hash(repo="/content/phishing-detection"):\n    try:\n        return subprocess.check_output(\n            ["git", "-C", repo, "rev-parse", "--short", "HEAD"],\n            stderr=subprocess.DEVNULL).decode().strip()\n    except Exception:\n        return "nogit"\n\n\nclass Checkpoint:\n    # Resumable checkpointing for Colab. Survives runtime resets via Drive.\n\n    def __init__(self, root="/content/phishing-detection/checkpoints",\n                 run_id=None, verbose=True):\n        self.root = Path(root)\n        self.run_id = run_id or time.strftime("run_%Y%m%d_%H%M%S")\n        self.dir = self.root / self.run_id\n        self.dir.mkdir(parents=True, exist_ok=True)\n        self.verbose = verbose\n        self.manifest_path = self.dir / "manifest.json"\n        self.manifest = (json.loads(self.manifest_path.read_text())\n                         if self.manifest_path.exists() else {})\n\n    @staticmethod\n    def _key(params):\n        if params is None:\n            return "default"\n        blob = json.dumps(params, sort_keys=True, default=str)\n        return hashlib.md5(blob.encode()).hexdigest()[:10]\n\n    def _path(self, stage, params=None):\n        return self.dir / (stage + "__" + self._key(params) + ".joblib")\n\n    def exists(self, stage, params=None):\n        return self._path(stage, params).exists()\n\n    def save(self, stage, obj, params=None, meta=None):\n        p = self._path(stage, params)\n        joblib.dump(obj, p, compress=3)\n        size_mb = round(p.stat().st_size / 1e6, 3)\n        self.manifest[p.name] = {\n            "stage": stage, "params": params, "meta": meta or {},\n            "git_commit": _git_hash(),\n            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "size_mb": size_mb,\n        }\n        self.manifest_path.write_text(\n            json.dumps(self.manifest, indent=2, default=str))\n        if self.verbose:\n            print("[ckpt] saved " + p.name + "  (" + str(size_mb) + " MB)")\n        return p\n\n    def load(self, stage, params=None):\n        p = self._path(stage, params)\n        if not p.exists():\n            raise FileNotFoundError("No checkpoint: " + str(p))\n        if self.verbose:\n            print("[ckpt] loaded " + p.name)\n        return joblib.load(p)\n\n    def cache(self, stage, fn, params=None, meta=None, force=False):\n        if self.exists(stage, params) and not force:\n            return self.load(stage, params)\n        if self.verbose:\n            print("[ckpt] computing " + stage + " ...")\n        out = fn()\n        self.save(stage, out, params=params, meta=meta)\n        return out\n\n    def summary(self):\n        import pandas as pd\n        if not self.manifest:\n            return pd.DataFrame()\n        return pd.DataFrame(self.manifest).T[\n            ["stage", "saved_at", "size_mb", "git_commit"]]\n\n    @classmethod\n    def latest_run(cls, root="/content/phishing-detection/checkpoints", **kw):\n        runs = sorted(p.name for p in Path(root).glob("run_*") if p.is_dir())\n        if not runs:\n            raise FileNotFoundError("No previous runs.")\n        return cls(root=root, run_id=runs[-1], **kw)\n'

target = REPO_DIR / "src" / "utils" / "checkpoint.py"
backup = DRIVE_DIR / "_repo_backup" / "src" / "utils" / "checkpoint.py"
if backup.exists():
    target.write_text(backup.read_text())
    print("restored checkpoint.py from Drive backup")
else:
    target.write_text(CKPT_SRC)
    print("wrote checkpoint.py from embedded source")

import importlib, src.utils.checkpoint as _ck
importlib.reload(_ck)
from src.utils.checkpoint import Checkpoint
print("Checkpoint OK")

restored checkpoint.py from Drive backup
Checkpoint OK


## Cell 2 — Load the harmonised data

Reads the checkpoint from notebook 02. If it is missing, rebuilds from the raw
CSVs on Drive — so this notebook still runs if you skipped straight here.

In [3]:
import pandas as pd, numpy as np

REGISTRY = {
    "phiusiil":       {"rows": 235795, "label": "label",       "phish": 0,
                       "url_col": "URL"},
    "uci2012":        {"rows": 11055,  "label": "Result",      "phish": -1,
                       "url_col": None},
    "uci_multiclass": {"rows": 1353,   "label": "Result",      "phish": -1,
                       "url_col": None},
    "hannousse":      {"rows": 11430,  "label": "status",      "phish": "phishing",
                       "url_col": "url"},
    "vrbancic":       {"rows": 88647,  "label": "phishing",    "phish": 1,
                       "url_col": None},
    "tan":            {"rows": 10000,  "label": "CLASS_LABEL", "phish": 1,
                       "url_col": None},
}

def _harmonise(key, df):
    spec = REGISTRY[key]
    col = [c for c in df.columns if c.lower() == spec["label"].lower()]
    if not col:
        raise KeyError(f"{key}: label column '{spec['label']}' not found")
    col = col[0]
    ser = df[col]
    if isinstance(spec["phish"], str):
        y = (ser.astype(str).str.lower() == spec["phish"].lower()).astype(int)
    else:
        y = (ser == spec["phish"]).astype(int)
    return df.drop(columns=[col]), y

ck = Checkpoint(run_id="data_acquisition")

def rebuild():
    out = {}
    for k in REGISTRY:
        p = RAW / k / f"{k}.csv"
        if not p.exists():
            print(f"  [skip] {k} - no CSV at {p}")
            continue
        df = pd.read_csv(p, low_memory=False)
        X, y = _harmonise(k, df)
        out[k] = {"X": X, "y": y}
    return out

try:
    bundle = ck.load("harmonised_v2",
                     params={"convention": "1=phishing",
                             "sources": sorted([k for k in REGISTRY
                                                if (RAW / k / f"{k}.csv").exists()])})
except Exception as e:
    print(f"checkpoint miss ({type(e).__name__}) - rebuilding from raw CSVs")
    bundle = rebuild()

for k, v in bundle.items():
    print(f"{k:16s} X={str(v['X'].shape):>14s}  phishing_rate={v['y'].mean():.4f}")

[ckpt] loaded harmonised_v2__8ab8683c29.joblib
phiusiil         X=  (235795, 54)  phishing_rate=0.4281
uci2012          X=   (11055, 30)  phishing_rate=0.4431
uci_multiclass   X=     (1353, 9)  phishing_rate=0.5188
hannousse        X=   (11430, 88)  phishing_rate=0.5000
vrbancic         X=  (88647, 111)  phishing_rate=0.3457


## Cell 3 — Canonical URL feature extractor

Writes `src/features/urlfeat.py`. Thirty-two lexical features computed **identically**
for any source that ships raw URLs. This is what makes cross-dataset comparison
legitimate: without a shared feature definition, training on one benchmark and
testing on another compares two different measurement instruments.

Note `registrable_domain` returns the bare IP for IP-literal hosts. Splitting
`192.168.1.44` into a pseudo-domain would put sibling URLs on both sides of a
grouped split and quietly reintroduce the leakage the split exists to prevent.

In [4]:
%%writefile /content/phishing-detection/src/features/urlfeat.py
"""Canonical URL-lexical features + split strategies. Tested standalone."""
import re, math
from urllib.parse import urlparse
import numpy as np
import pandas as pd

SUSPICIOUS_WORDS = [
    "login", "signin", "verify", "account", "update", "secure", "webscr",
    "banking", "confirm", "password", "credential", "invoice", "payment",
    "billing", "suspend", "unlock", "recover", "wallet", "support",
]
SHORTENERS = {
    "bit.ly", "goo.gl", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly",
    "adf.ly", "bit.do", "cutt.ly", "rb.gy", "shorte.st", "rebrand.ly",
}
IPV4 = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")
HEX_HOST = re.compile(r"^0x[0-9a-f]+$", re.I)


def _entropy(s):
    if not s:
        return 0.0
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())


def canonical_url_features(url, extractor=None):
    """~30 lexical features computable from a URL alone, identically for any source."""
    url = str(url).strip()
    if "://" not in url:
        url = "http://" + url
    try:
        p = urlparse(url)
    except Exception:
        p = urlparse("http://invalid")

    host = (p.hostname or "").lower()
    path = p.path or ""
    query = p.query or ""

    if extractor is not None:
        ext = extractor(url)
        subdomain, domain, suffix = ext.subdomain, ext.domain, ext.suffix
    else:
        parts = host.split(".")
        suffix = parts[-1] if len(parts) > 1 else ""
        domain = parts[-2] if len(parts) > 2 else (parts[0] if parts else "")
        subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""

    registrable = f"{domain}.{suffix}".strip(".")
    digits = sum(ch.isdigit() for ch in url)
    letters = sum(ch.isalpha() for ch in url)

    f = {
        "url_length": len(url),
        "host_length": len(host),
        "path_length": len(path),
        "query_length": len(query),
        "n_dots": url.count("."),
        "n_hyphens": url.count("-"),
        "n_underscores": url.count("_"),
        "n_slashes": path.count("/"),
        "n_question": url.count("?"),
        "n_equals": url.count("="),
        "n_at": url.count("@"),
        "n_ampersand": url.count("&"),
        "n_percent": url.count("%"),
        "n_digits": digits,
        "n_letters": letters,
        "digit_ratio": digits / max(len(url), 1),
        "n_subdomains": len([s for s in subdomain.split(".") if s]),
        "subdomain_length": len(subdomain),
        "domain_length": len(domain),
        "tld_length": len(suffix),
        "is_ip_host": int(bool(IPV4.match(host))),
        "is_hex_host": int(bool(HEX_HOST.match(host))),
        "is_https": int(p.scheme == "https"),
        "has_port": int(p.port is not None),
        "is_shortener": int(registrable in SHORTENERS),
        "has_punycode": int("xn--" in host),
        "has_double_slash_path": int("//" in path),
        "n_suspicious_words": sum(w in url.lower() for w in SUSPICIOUS_WORDS),
        "url_entropy": _entropy(url),
        "host_entropy": _entropy(host),
        "longest_token_path": max([len(t) for t in re.split(r"[/\-_.]", path) if t] or [0]),
        "has_https_token_in_host": int("https" in host),
    }
    return f


def build_canonical_matrix(urls, use_tldextract=True):
    extractor = None
    if use_tldextract:
        try:
            import tldextract
            extractor = tldextract.TLDExtract(suffix_list_urls=())
        except Exception:
            extractor = None
    rows = [canonical_url_features(u, extractor) for u in urls]
    return pd.DataFrame(rows, index=getattr(urls, "index", None))


_DEFAULT_EXTRACTOR = None


def get_extractor():
    """Cached tldextract instance using the bundled PSL snapshot (no network)."""
    global _DEFAULT_EXTRACTOR
    if _DEFAULT_EXTRACTOR is None:
        try:
            import tldextract
            _DEFAULT_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())
        except Exception:
            _DEFAULT_EXTRACTOR = False
    return _DEFAULT_EXTRACTOR or None


def registrable_domain(url, extractor="auto"):
    """eTLD+1. Returns the bare IP for IP-literal hosts, which must not be
    split into pseudo-domains or grouped splitting will leak."""
    url = str(url)
    if "://" not in url:
        url = "http://" + url
    host = (urlparse(url).hostname or "").lower()
    if not host:
        return ""
    if IPV4.match(host):
        return host
    if extractor == "auto":
        extractor = get_extractor()
    if extractor is not None:
        e = extractor(url)
        reg = f"{e.domain}.{e.suffix}".strip(".")
        if reg:
            return reg
    parts = host.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else host


# ------------------------------------------------------------------ splits
def grouped_split(X, y, groups, test_size=0.2, seed=42):
    """No registrable domain appears in both train and test."""
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr, te = next(gss.split(X, y, groups))
    return tr, te


def stratified_split(X, y, test_size=0.2, seed=42):
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(y))
    return train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)


def rebalance_to_base_rate(y, target_rate, seed=42, index=None):
    """Downsample the phishing class to a realistic prevalence."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y)) if index is None else np.asarray(index)
    pos = idx[np.asarray(y) == 1]
    neg = idx[np.asarray(y) == 0]
    n_pos = int(round(target_rate * len(neg) / (1 - target_rate)))
    n_pos = min(n_pos, len(pos))
    keep = np.concatenate([neg, rng.choice(pos, size=n_pos, replace=False)])
    rng.shuffle(keep)
    return keep

Writing /content/phishing-detection/src/features/urlfeat.py


## Cell 4 — Leakage audit

For every feature in every dataset, fit a depth-3 stump on that feature alone and
record cross-validated AUC. Anything above 0.95 is encoding the label, not
describing the URL.

This is the single most useful diagnostic in the notebook. Run it before you
believe any headline accuracy.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

def leakage_audit(X, y, max_rows=30000, seed=42, threshold=0.95):
    Xn = X.select_dtypes(include=[np.number])
    if len(Xn) > max_rows:
        idx = np.random.default_rng(seed).choice(len(Xn), max_rows, replace=False)
        Xn, ys = Xn.iloc[idx], np.asarray(y)[idx]
    else:
        ys = np.asarray(y)
    rows = []
    for col in Xn.columns:
        v = Xn[[col]].fillna(-999)
        if v[col].nunique() < 2:
            continue
        try:
            auc = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=seed),
                                  v, ys, cv=3, scoring="roc_auc").mean()
        except Exception:
            continue
        rows.append({"feature": col, "single_feature_auc": round(float(auc), 4)})
    df = pd.DataFrame(rows).sort_values("single_feature_auc", ascending=False)
    df["flag"] = np.where(df["single_feature_auc"] >= threshold, "LEAKY", "")
    return df

audits = {}
for k, v in bundle.items():
    print(f"\n=== {k} ===")
    a = leakage_audit(v["X"], v["y"])
    audits[k] = a
    print(a.head(8).to_string(index=False))
    n_leaky = (a["flag"] == "LEAKY").sum()
    print(f"  -> {n_leaky} feature(s) at or above 0.95 AUC")


=== phiusiil ===
           feature  single_feature_auc  flag
URLSimilarityIndex              0.9963 LEAKY
        LineOfCode              0.9885 LEAKY
   NoOfExternalRef              0.9877 LEAKY
         NoOfImage              0.9809 LEAKY
            NoOfJS              0.9734 LEAKY
       NoOfSelfRef              0.9710 LEAKY
           NoOfCSS              0.9541 LEAKY
      HasSocialNet              0.8968      
  -> 7 feature(s) at or above 0.95 AUC

=== uci2012 ===
                   feature  single_feature_auc flag
            sslfinal_state              0.8942     
             url_of_anchor              0.8759     
         having_sub_domain              0.6973     
               web_traffic              0.6752     
             links_in_tags              0.6364     
               request_url              0.6253     
             prefix_suffix              0.6190     
domain_registration_length              0.6070     
  -> 0 feature(s) at or above 0.95 AUC

=== uci_multi

**Reading the output.** A handful of features in the 0.80–0.90 band is normal
and healthy — those are genuinely predictive signals like `n_subdomains` or SSL
state. A single feature at 0.99+ is not a signal, it is the label wearing a
disguise. Drop those and report both conditions in the paper; the honest
comparison is itself a contribution given how many published results never check.

In [6]:
DROP = {k: list(a.loc[a["flag"] == "LEAKY", "feature"]) for k, a in audits.items()}
for k, cols in DROP.items():
    print(f"{k:16s} dropping: {cols if cols else '(none)'}")

clean = {}
for k, v in bundle.items():
    Xc = v["X"].drop(columns=DROP[k], errors="ignore")
    clean[k] = {"X": Xc, "y": v["y"]}
    print(f"{k:16s} {v['X'].shape[1]:>3d} -> {Xc.shape[1]:>3d} features")

(REPO_DIR / "reports" / "tables").mkdir(parents=True, exist_ok=True)
pd.concat([a.assign(dataset=k) for k, a in audits.items()]).to_csv(
    REPO_DIR / "reports" / "tables" / "leakage_audit.csv", index=False)
print("\nsaved -> reports/tables/leakage_audit.csv")

phiusiil         dropping: ['URLSimilarityIndex', 'LineOfCode', 'NoOfExternalRef', 'NoOfImage', 'NoOfJS', 'NoOfSelfRef', 'NoOfCSS']
uci2012          dropping: (none)
uci_multiclass   dropping: (none)
hannousse        dropping: (none)
vrbancic         dropping: (none)
phiusiil          54 ->  47 features
uci2012           30 ->  30 features
uci_multiclass     9 ->   9 features
hannousse         88 ->  88 features
vrbancic         111 -> 111 features

saved -> reports/tables/leakage_audit.csv


## Cell 5 — Which datasets can participate in cross-dataset work

Only sources shipping raw URLs can be projected into the canonical feature space.
The rest are pre-extracted feature matrices with incompatible, undocumented
definitions — a column called `having_IP_Address` in one dataset is not
necessarily computed the same way as in another.

In [7]:
import importlib
import src.features.urlfeat as uf
importlib.reload(uf)

url_capable, feature_only = [], []
for k, v in clean.items():
    spec = REGISTRY[k]
    col = spec["url_col"]
    present = col and any(c.lower() == col.lower() for c in v["X"].columns)
    (url_capable if present else feature_only).append(k)

print("canonical-space capable :", url_capable)
print("feature-matrix only     :", feature_only)
print()
print("Cross-dataset experiments use the canonical space and are limited to the")
print("first group. The second group supports within-dataset results only.")

canonical-space capable : ['phiusiil', 'hannousse']
feature-matrix only     : ['uci2012', 'uci_multiclass', 'vrbancic']

Cross-dataset experiments use the canonical space and are limited to the
first group. The second group supports within-dataset results only.


## Cell 6 — Build canonical matrices

In [8]:
def canonical_for(key):
    spec = REGISTRY[key]
    X = clean[key]["X"]
    col = [c for c in X.columns if c.lower() == spec["url_col"].lower()][0]
    urls = X[col].astype(str)
    M = uf.build_canonical_matrix(urls)
    dom = urls.map(uf.registrable_domain)
    return M, clean[key]["y"].reset_index(drop=True), dom.reset_index(drop=True), urls

canon = {}
for k in url_capable:
    M, y, dom, urls = canonical_for(k)
    canon[k] = {"X": M, "y": y, "domain": dom, "url": urls}
    print(f"{k:12s} canonical X={M.shape}  unique domains={dom.nunique():,}  "
          f"phishing_rate={y.mean():.4f}")

phiusiil     canonical X=(235795, 32)  unique domains=175,509  phishing_rate=0.4281
hannousse    canonical X=(11430, 32)  unique domains=6,635  phishing_rate=0.5000


## Cell 7 — Three split strategies

| Strategy | What it tests | Weakness |
|---|---|---|
| `stratified` | Nothing beyond memorisation | Sibling URLs from one kit land on both sides |
| `grouped` | Generalisation to unseen hosts | Still same-source, same-era |
| `cross_dataset` | Generalisation to a different collection | Confounds era, method and source |

Report all three. The **gap between stratified and grouped is your leakage
estimate**, and it is a result worth a table on its own.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42

def evaluate(Xtr, ytr, Xte, yte, seed=SEED):
    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed)
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    return {
        "roc_auc": round(float(roc_auc_score(yte, p)), 4),
        "pr_auc":  round(float(average_precision_score(yte, p)), 4),
        "f1":      round(float(f1_score(yte, (p >= 0.5).astype(int))), 4),
    }

rows = []
for k, d in canon.items():
    X, y, dom = d["X"], d["y"], d["domain"]

    tr, te = uf.stratified_split(X, y, seed=SEED)
    r = evaluate(X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te])
    rows.append({"dataset": k, "split": "stratified", **r})

    tr, te = uf.grouped_split(X, y, dom, seed=SEED)
    r = evaluate(X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te])
    rows.append({"dataset": k, "split": "grouped", **r})

split_df = pd.DataFrame(rows)
print(split_df.to_string(index=False))

piv = split_df.pivot(index="dataset", columns="split", values="roc_auc")
if {"stratified", "grouped"}.issubset(piv.columns):
    piv["leakage_gap"] = (piv["stratified"] - piv["grouped"]).round(4)
    print("\nLeakage gap (stratified minus grouped ROC-AUC):")
    print(piv.to_string())

  dataset      split  roc_auc  pr_auc     f1
 phiusiil stratified   0.9986  0.9986 0.9973
 phiusiil    grouped   0.9978  0.9975 0.9959
hannousse stratified   0.9655  0.9686 0.9042
hannousse    grouped   0.9394  0.9220 0.8608

Leakage gap (stratified minus grouped ROC-AUC):
split      grouped  stratified  leakage_gap
dataset                                    
hannousse   0.9394      0.9655       0.0261
phiusiil    0.9978      0.9986       0.0008


## Cell 8 — Cross-dataset harness

Train on one source, test on every other, in the shared canonical space. Expect a
substantial drop. That drop is the paper's central empirical claim — not the
within-dataset number everyone else reports.

In [10]:
cross_rows = []
for src in canon:
    Xs, ys = canon[src]["X"], canon[src]["y"]
    for dst in canon:
        if src == dst:
            tr, te = uf.grouped_split(Xs, ys, canon[src]["domain"], seed=SEED)
            r = evaluate(Xs.iloc[tr], ys.iloc[tr], Xs.iloc[te], ys.iloc[te])
            r["note"] = "within (grouped)"
        else:
            Xd, yd = canon[dst]["X"], canon[dst]["y"]
            cols = [c for c in Xs.columns if c in Xd.columns]
            r = evaluate(Xs[cols], ys, Xd[cols], yd)
            r["note"] = "cross"
        cross_rows.append({"train": src, "test": dst, **r})

cross_df = pd.DataFrame(cross_rows)
print(cross_df.to_string(index=False))

print("\nROC-AUC matrix (rows = train, cols = test):")
print(cross_df.pivot(index="train", columns="test", values="roc_auc").to_string())

cross_df.to_csv(REPO_DIR / "reports" / "tables" / "cross_dataset.csv", index=False)

    train      test  roc_auc  pr_auc     f1             note
 phiusiil  phiusiil   0.9978  0.9975 0.9959 within (grouped)
 phiusiil hannousse   0.5169  0.5088 0.6674            cross
hannousse  phiusiil   0.8695  0.8830 0.7751            cross
hannousse hannousse   0.9394  0.9220 0.8608 within (grouped)

ROC-AUC matrix (rows = train, cols = test):
test       hannousse  phiusiil
train                         
hannousse     0.9394    0.8695
phiusiil      0.5169    0.9978


**Watch the F1 column, not just AUC.** In cross-dataset runs it is common to
see ROC-AUC stay high while F1 collapses toward zero. That is not a broken model
— it means the *ranking* transfers but the decision *threshold* does not. The
score distribution shifts between collections, so 0.5 stops being the right cut.

Two consequences for the paper:

- Report threshold-free metrics (ROC-AUC, PR-AUC) **and** threshold-dependent ones
  (F1, precision, recall) side by side. The divergence is itself the finding.
- Any deployable claim needs threshold recalibration on the target distribution.
  Say so explicitly — a reviewer who has done deployment work will look for it.

## Cell 9 — Benign-source bias probe

The hidden confound: if benign URLs come from popular-site lists and phishing
URLs from abuse feeds, a classifier can score well by learning *popular versus
obscure*. This cell checks how many benign hosts appear in Tranco's head, and
whether the model's confidence tracks popularity rank.

If benign instances are overwhelmingly high-rank, the paper needs a long-tail
benign sample before the results mean anything.

In [11]:
tranco_path = REPO_DIR / "data" / "external" / "tranco" / "tranco_top1m.csv"
if not tranco_path.exists():
    print("Tranco list not found - run notebook 02 Cell 3 first. Skipping probe.")
else:
    tr_df = pd.read_csv(tranco_path, names=["rank", "domain"])
    rank_of = dict(zip(tr_df["domain"], tr_df["rank"]))

    for k, d in canon.items():
        dom, y = d["domain"], d["y"]
        benign = dom[y == 0]
        ranks = benign.map(rank_of)
        found = ranks.notna()
        print(f"\n=== {k} ===")
        print(f"  benign hosts in Tranco 1M : {found.mean():.1%}")
        if found.any():
            r = ranks[found]
            for band, lo, hi in [("top 1k", 0, 1_000), ("1k-10k", 1_000, 10_000),
                                 ("10k-100k", 10_000, 100_000),
                                 ("100k-1M", 100_000, 1_000_000)]:
                share = ((r > lo) & (r <= hi)).mean()
                print(f"    {band:>10s}: {share:6.1%}")
            print(f"  median rank: {r.median():,.0f}")
        print(f"  phishing hosts in Tranco  : "
              f"{dom[y == 1].map(rank_of).notna().mean():.1%}")


=== phiusiil ===
  benign hosts in Tranco 1M : 33.3%
        top 1k:   0.9%
        1k-10k:   6.6%
      10k-100k:  22.8%
       100k-1M:  69.7%
  median rank: 226,379
  phishing hosts in Tranco  : 45.3%

=== hannousse ===
  benign hosts in Tranco 1M : 64.5%
        top 1k:  27.2%
        1k-10k:  18.3%
      10k-100k:  26.2%
       100k-1M:  28.4%
  median rank: 12,820
  phishing hosts in Tranco  : 32.8%


**How to read it.** If most benign hosts sit in the top 10k while phishing
hosts are absent from the list entirely, popularity alone separates the classes.
The fix is not a better model — it is sampling benign URLs from ranks 100k–1M so
that obscurity stops being a proxy for the label. State the benign rank
distribution in the paper; almost nobody does, and it is a fair question at review.

## Cell 10 — Realistic base-rate evaluation set

Benchmarks are near-balanced; real traffic is not. Precision at a 50% prior tells
you nothing about precision at 0.1%. This builds evaluation sets at several
prevalences and shows what happens to precision while ROC-AUC barely moves.

In [12]:
from sklearn.metrics import precision_score, recall_score

k = list(canon)[0]
X, y, dom = canon[k]["X"], canon[k]["y"], canon[k]["domain"]
tr, te = uf.grouped_split(X, y, dom, seed=SEED)

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)
model.fit(X.iloc[tr], y.iloc[tr])

Xte, yte = X.iloc[te].reset_index(drop=True), y.iloc[te].reset_index(drop=True)
rows = []
for rate in [0.5, 0.1, 0.01, 0.001]:
    keep = uf.rebalance_to_base_rate(yte, rate, seed=SEED)
    if len(keep) < 50 or yte.iloc[keep].sum() < 5:
        rows.append({"base_rate": rate, "n": len(keep), "note": "too few positives"})
        continue
    p = model.predict_proba(Xte.iloc[keep])[:, 1]
    yy = yte.iloc[keep]
    pred = (p >= 0.5).astype(int)
    rows.append({
        "base_rate": rate,
        "n": len(keep),
        "actual_rate": round(float(yy.mean()), 5),
        "roc_auc": round(float(roc_auc_score(yy, p)), 4),
        "pr_auc": round(float(average_precision_score(yy, p)), 4),
        "precision": round(float(precision_score(yy, pred, zero_division=0)), 4),
        "recall": round(float(recall_score(yy, pred, zero_division=0)), 4),
    })

base_df = pd.DataFrame(rows)
print(f"Dataset: {k}")
print(base_df.to_string(index=False))
print("\nROC-AUC is nearly flat across prevalence; precision collapses.")
print("This is why the paper reports PR-AUC and precision at a stated base rate.")
base_df.to_csv(REPO_DIR / "reports" / "tables" / "base_rate_sensitivity.csv", index=False)

Dataset: phiusiil
 base_rate     n  actual_rate  roc_auc  pr_auc  precision  recall
     0.500 41952      0.35769   0.9978  0.9975     0.9983  0.9936
     0.100 29940      0.10000   0.9986  0.9974     0.9914  0.9957
     0.010 27218      0.00999   0.9915  0.9851     0.9113  0.9816
     0.001 26973      0.00100   0.9787  0.9630     0.5000  0.9630

ROC-AUC is nearly flat across prevalence; precision collapses.
This is why the paper reports PR-AUC and precision at a stated base rate.


## Cell 11 — Persist splits and checkpoint

In [13]:
splits = {}
for k, d in canon.items():
    X, y, dom = d["X"], d["y"], d["domain"]
    s_tr, s_te = uf.stratified_split(X, y, seed=SEED)
    g_tr, g_te = uf.grouped_split(X, y, dom, seed=SEED)
    splits[k] = {
        "stratified": {"train": s_tr, "test": s_te},
        "grouped":    {"train": g_tr, "test": g_te},
    }
    print(f"{k:12s} stratified {len(s_tr):>7,}/{len(s_te):<7,}  "
          f"grouped {len(g_tr):>7,}/{len(g_te):<7,}")

ck3 = Checkpoint(run_id="features_and_splits")
ck3.save("canonical_matrices",
         {k: {"X": v["X"], "y": v["y"], "domain": v["domain"]} for k, v in canon.items()},
         params={"feature_set": "canonical_v1", "seed": SEED},
         meta={"n_features": int(list(canon.values())[0]["X"].shape[1])})
ck3.save("splits", splits,
         params={"strategies": ["stratified", "grouped"], "seed": SEED},
         meta={"note": "no chronological split - datasets carry no timestamps"})
ck3.save("leakage_audit", audits, params={"threshold": 0.95})
ck3.summary()

phiusiil     stratified 188,636/47,159   grouped 193,843/41,952 
hannousse    stratified   9,144/2,286    grouped   9,491/1,939  
[ckpt] saved canonical_matrices__af841610a7.joblib  (8.718 MB)
[ckpt] saved splits__470e7cc5bf.joblib  (1.134 MB)
[ckpt] saved leakage_audit__52a4155916.joblib  (0.005 MB)


,stage,saved_at,size_mb,git_commit
canonical_matrices__af841610a7.joblib,canonical_matrices,2026-07-24 04:42:46,8.718,nogit
splits__470e7cc5bf.joblib,splits,2026-07-24 04:42:47,1.134,nogit
leakage_audit__52a4155916.joblib,leakage_audit,2026-07-24 04:42:47,0.005,nogit


## Cell 12 — Sync and commit

In [14]:
import shutil, glob, subprocess

for src in glob.glob(str(DRIVE_DIR / "notebooks" / "*.ipynb")):
    shutil.copy2(src, REPO_DIR / "notebooks")
for folder in ["src", "config", "reports"]:
    dst = DRIVE_DIR / "_repo_backup" / folder
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(REPO_DIR / folder, dst)

subprocess.run(["git", "-C", str(REPO_DIR), "add", "-A"])
r = subprocess.run(["git", "-C", str(REPO_DIR), "commit", "-m",
                    "feat: canonical URL features, leakage audit, split strategies"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

fatal: not a git repository (or any of the parent directories): .git



---
### What to check before notebook 04

1. **Leakage audit** — any feature above 0.95 AUC is dropped and reported, not silently removed.
2. **Leakage gap** — the stratified-minus-grouped difference. If it is near zero, your grouping key may be wrong.
3. **Cross-dataset matrix** — the off-diagonal drop is the paper's headline.
4. **Benign rank distribution** — if benign hosts cluster in Tranco's top 10k, note it as a limitation or resample.

Next: `04_models_and_evaluation.ipynb` — model comparison under the grouped and
cross-dataset protocols, cost-sensitive thresholds, and SHAP attribution.